# 👁️ ARGOS Universal OS — One-Click Colab Launch

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/labuaqlysnecy/Argoss/blob/main/colab/ARGOS_Colab_Launch.ipynb)

This notebook starts **ARGOS** in headless server mode inside Google Colab and exposes the HTTP Remote Control API via a public Cloudflare Tunnel URL.

## Required secrets (set via Colab ▶ Secrets panel or environment variables)
| Variable | Required | Description |
|---|---|---|
| `ARGOS_REMOTE_TOKEN` | ✅ | Bearer token for API authentication |
| `TELEGRAM_BOT_TOKEN` | ⬜ optional | Telegram bot token (leave empty to disable) |
| `USER_ID` | ⬜ optional | Your Telegram user ID |

## Steps
1. Add secrets above in **Colab → Secrets** (🔑 icon in the left sidebar)
2. Run **all cells** (Runtime → Run all)
3. Copy the public URL printed in the last cell and enter it in the Android APK Settings tab

In [ ]:
# ── Cell 1: Clone ARGOS repository ──────────────────────────────────────────
import os

REPO_DIR = "/content/Argoss"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/labuaqlysnecy/Argoss.git {REPO_DIR}
else:
    print(f"✅ Repo already cloned at {REPO_DIR}")
    !cd {REPO_DIR} && git pull --ff-only

%cd {REPO_DIR}
print("✅ Working directory:", os.getcwd())

In [ ]:
# ── Cell 2: Install system dependencies (headless, no GUI) ──────────────────
# APK build tooling is OFF by default — set BUILD_APK=True to enable.
BUILD_APK = False

!apt-get update -qq
!apt-get install -y -qq \
    libsqlite3-dev \
    libssl-dev \
    curl wget \
    2>/dev/null

if BUILD_APK:
    print("🔧 Installing APK build tooling (buildozer deps)...")
    !apt-get install -y -qq \
        openjdk-17-jdk \
        autoconf automake libtool \
        zip unzip \
        libffi-dev \
        2>/dev/null
    !pip install -q buildozer cython

print("✅ System deps installed.")

In [ ]:
# ── Cell 3: Install Python dependencies ─────────────────────────────────────
!pip install -q \
    fastapi \
    uvicorn[standard] \
    python-dotenv \
    psutil \
    requests \
    paho-mqtt

# Try installing optional heavy deps; ignore failures (Colab may already have them)
!pip install -q aiogram 2>/dev/null || true

print("✅ Python deps installed.")

In [ ]:
# ── Cell 4: Load secrets ────────────────────────────────────────────────────
import os

def _secret(name: str, default: str = "") -> str:
    """Try Colab userdata first, then environment variable."""
    # Try Colab secrets
    try:
        from google.colab import userdata
        val = userdata.get(name)
        if val:
            return val
    except Exception:
        pass
    # Fall back to environment variable
    return os.environ.get(name, default)

ARGOS_REMOTE_TOKEN   = _secret("ARGOS_REMOTE_TOKEN")
TELEGRAM_BOT_TOKEN   = _secret("TELEGRAM_BOT_TOKEN")
USER_ID              = _secret("USER_ID")
ARGOS_API_PORT       = int(os.environ.get("ARGOS_API_PORT", "8080"))

# Inject into environment so ARGOS picks them up
os.environ["ARGOS_REMOTE_TOKEN"] = ARGOS_REMOTE_TOKEN
if TELEGRAM_BOT_TOKEN:
    os.environ["TELEGRAM_BOT_TOKEN"] = TELEGRAM_BOT_TOKEN
if USER_ID:
    os.environ["USER_ID"] = USER_ID

if not ARGOS_REMOTE_TOKEN:
    print("⚠️  WARNING: ARGOS_REMOTE_TOKEN is not set. "
          "The API will be accessible without authentication.")
else:
    print("✅ ARGOS_REMOTE_TOKEN loaded (hidden).")

if TELEGRAM_BOT_TOKEN:
    print("✅ TELEGRAM_BOT_TOKEN loaded.")
else:
    print("ℹ️  Telegram disabled (no TELEGRAM_BOT_TOKEN).")

In [ ]:
# ── Cell 5: Start ARGOS in headless mode with dashboard ─────────────────────
# We start only the core + FastAPI dashboard (no GUI, safe Telegram handling)
import sys, os, threading, time

sys.path.insert(0, "/content/Argoss")
os.chdir("/content/Argoss")

# Create required directories
os.makedirs("logs", exist_ok=True)
os.makedirs("data", exist_ok=True)

# Write minimal .env so ARGOS doesn't crash on missing vars
env_lines = []
if ARGOS_REMOTE_TOKEN:
    env_lines.append(f"ARGOS_REMOTE_TOKEN={ARGOS_REMOTE_TOKEN}")
if TELEGRAM_BOT_TOKEN:
    env_lines.append(f"TELEGRAM_BOT_TOKEN={TELEGRAM_BOT_TOKEN}")
if USER_ID:
    env_lines.append(f"USER_ID={USER_ID}")

with open(".env", "w") as f:
    f.write("\n".join(env_lines))

print("✅ .env written.")
print(f"🚀 Starting ARGOS on port {ARGOS_API_PORT} ...")

# ── Start ARGOS headless ─────────────────────────────────────────────────
_argos_ready = threading.Event()
_argos_error = []

def _start_argos():
    try:
        from dotenv import load_dotenv
        load_dotenv()

        from src.core import ArgosCore
        from src.admin import ArgosAdmin
        from src.factory.flasher import AirFlasher
        from src.interface.fastapi_dashboard import FastAPIDashboard

        # Suppress Telegram in Colab (set_wakeup_fd only works in main thread)
        os.environ.setdefault("ARGOS_NO_TELEGRAM", "1")

        core    = ArgosCore()
        admin   = ArgosAdmin()
        flasher = AirFlasher()

        dash = FastAPIDashboard(core, admin, flasher, port=ARGOS_API_PORT)
        result = dash.start()
        print(result)
        _argos_ready.set()

        # Keep alive
        while True:
            time.sleep(60)
    except Exception as exc:
        import traceback
        _argos_error.append(str(exc))
        traceback.print_exc()
        _argos_ready.set()

_t = threading.Thread(target=_start_argos, daemon=True)
_t.start()

# Wait up to 30 s for ARGOS to come up
_argos_ready.wait(timeout=30)

if _argos_error:
    print(f"❌ ARGOS failed to start: {_argos_error[0]}")
else:
    print("✅ ARGOS started successfully.")

In [ ]:
# ── Cell 6: Start Cloudflare Tunnel (cloudflared) ───────────────────────────
import subprocess, threading, re, time

# Download cloudflared if not present
CF_BIN = "/usr/local/bin/cloudflared"

if not os.path.exists(CF_BIN):
    print("📥 Downloading cloudflared...")
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 \
        -O {CF_BIN}
    !chmod +x {CF_BIN}
    print("✅ cloudflared installed.")

PUBLIC_URL = []

def _run_tunnel():
    cmd = [CF_BIN, "tunnel", "--url", f"http://localhost:{ARGOS_API_PORT}",
           "--no-autoupdate", "--logfile", "/tmp/cf_tunnel.log"]
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT,
                             universal_newlines=True)
    for line in proc.stdout:
        m = re.search(r'(https://[\w-]+\.trycloudflare\.com)', line)
        if m:
            PUBLIC_URL.append(m.group(1))
            break

_tunnel_thread = threading.Thread(target=_run_tunnel, daemon=True)
_tunnel_thread.start()

# Wait up to 20 s for URL
for _ in range(40):
    if PUBLIC_URL:
        break
    time.sleep(0.5)

if PUBLIC_URL:
    print(f"\n✅ Tunnel active!")
    print(f"\n🌐 Public base URL:\n   {PUBLIC_URL[0]}")
    print(f"\n📱 Enter this in the Android APK Settings tab as 'Server URL'.")
else:
    print("⚠️  Tunnel URL not detected. Check /tmp/cf_tunnel.log")
    !cat /tmp/cf_tunnel.log 2>/dev/null | tail -20

In [ ]:
# ── Cell 7: Health check ─────────────────────────────────────────────────────
import requests, json

base = PUBLIC_URL[0] if PUBLIC_URL else f"http://localhost:{ARGOS_API_PORT}"
headers = {"Authorization": f"Bearer {ARGOS_REMOTE_TOKEN}"} if ARGOS_REMOTE_TOKEN else {}

try:
    r = requests.get(f"{base}/api/health", timeout=10)
    print("GET /api/health →", r.status_code)
    print(json.dumps(r.json(), indent=2, ensure_ascii=False))
except Exception as e:
    print(f"❌ Health check failed: {e}")

try:
    r = requests.post(
        f"{base}/api/command",
        json={"cmd": "статус"},
        headers=headers,
        timeout=15,
    )
    print("\nPOST /api/command →", r.status_code)
    print(json.dumps(r.json(), indent=2, ensure_ascii=False))
except Exception as e:
    print(f"❌ Command test failed: {e}")

print("\n─" * 40)
print("Sample curl commands:")
print(f'  curl {base}/api/health')
print(f'  curl -X POST {base}/api/command \\')
print(f'       -H "Authorization: Bearer $ARGOS_REMOTE_TOKEN" \\')
print(f'       -H "Content-Type: application/json" \\')
print(f'       -d \'{{"cmd":"статус"}}\')')
print(f'  curl "{base}/api/events?limit=10" \\')
print(f'       -H "Authorization: Bearer $ARGOS_REMOTE_TOKEN"')